In [1]:
import os, numpy as np, torch
from torch.utils.data import Dataset, DataLoader

from cxt.utils import TIMES

def discretize(sequence, population_time):
    indices = np.searchsorted(population_time, sequence, side="right") - 1
    indices = np.clip(indices, 0, len(population_time) - 1)
    return indices  # keep as numpy array; convert to list if you really need it

class PairDataset(Dataset):
    """
    Each item is ONE pair from ONE TS.
    We prebuild a global, file-major index of (ts_path, pair_idx).
    Call `shuffle_files(seed)` between epochs to randomize FILE order
    (pairs within each file remain contiguous for fast I/O).
    """
    def __init__(self, root, split="train", mmap=True, shuffle_files=False, seed=1234):
        self.root = root
        self.split = split
        self.mmap = mmap

        # 1) discover files (X.npy/y.npy pairs) -> store as list of (X_path, y_path, P)
        split_dir = os.path.join(root, split)
        files = []
        for dirpath, _, filenames in os.walk(split_dir):
            if "X.npy" in filenames and "y.npy" in filenames:
                X_path = os.path.join(dirpath, "X.npy")
                y_path = os.path.join(dirpath, "y.npy")
                # cheap peek to get P without loading full array
                Y = np.load(y_path, mmap_mode="r") if mmap else np.load(y_path)
                P = int(Y.shape[0])
                if mmap:
                    del Y
                files.append((X_path, y_path, P))

        # keep the file list; optionally shuffle once at construction
        self._files = files
        if shuffle_files and len(self._files) > 1:
            rng = np.random.default_rng(int(seed))
            rng.shuffle(self._files)

        # 2) build the item index in FILE-MAJOR order
        self._build_items()

    def _build_items(self):
        items = []
        for X_path, y_path, P in self._files:
            # append all pairs for this file contiguously (fast locality)
            for p_idx in range(P):
                items.append((X_path, y_path, p_idx))
        self.items = items

    def shuffle_files(self, seed=None):
        """
        Shuffle **file order only** (not pairs inside a file) and rebuild the index.
        Call once per epoch for cheap mixing without killing locality.
        """
        if len(self._files) <= 1:
            return
        if seed is None:
            np.random.shuffle(self._files)
        else:
            rng = np.random.default_rng(int(seed))
            rng.shuffle(self._files)
        self._build_items()

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        X_path, y_path, p_idx = self.items[i]
        X = np.load(X_path, mmap_mode="r") if self.mmap else np.load(X_path)
        y = np.load(y_path, mmap_mode="r") if self.mmap else np.load(y_path)

        # feature slice (keep float16 from disk), then log1p in GPU later if you prefer
        Xi = torch.tensor(X[p_idx])            # (2, n_multipliers, n_steps, num_samples)
        Xi = torch.log1p(Xi)

        # label discretize -> tokens (BOS=1, +2 offset)
        yi = torch.tensor(y[p_idx])
        yi = torch.tensor(discretize(yi, TIMES)).long() + 2
        yi = torch.cat([torch.tensor([1]), yi])

        return Xi, yi


In [2]:
import numpy as np, torch
from torch.utils.data import IterableDataset, DataLoader

class ShuffleBufferDataset(IterableDataset):
    """
    Wraps an indexable dataset `ds` and yields items in near-random order
    using a small FIFO buffer. Reads from disk remain sequential (fast).
    """
    def __init__(self, ds, buffer_size: int = 8192, seed: int = 1234):
        super().__init__()
        self.ds = ds
        self.buffer_size = int(buffer_size)
        self.seed = int(seed)

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        N = len(self.ds)
        # iterate **sequentially** for disk locality
        src = iter(range(N))
        # fill buffer
        buf = []
        for _ in range(min(self.buffer_size, N)):
            i = next(src, None)
            if i is None: break
            buf.append(i)
        # stream: sample random from buffer, then refill with next sequential index
        while buf:
            j = int(rng.integers(0, len(buf)))
            idx = buf.pop(j)
            yield self.ds[idx]
            nxt = next(src, None)
            if nxt is not None:
                buf.append(nxt)


In [3]:
# your PairDataset from your last message
ds_base = PairDataset(root="/sietch_colab/kkor/cxt/ts/processed", split="train")
#ds_base.shuffle_files(seed=1234)   


# choose a buffer size that fits RAM (~few thousands to tens of thousands)
ds = ShuffleBufferDataset(ds_base, buffer_size=4096*8, seed=1234)


# single-process loader to preserve perfect locality; no sampler/shuffle here
loader = DataLoader(
    ds,
    batch_size=196,
    shuffle=False,        # ignored for IterableDataset
    num_workers=16,        # 0 is fastest on NFS/HDD
    pin_memory=False,
    drop_last=True,
)

In [ ]:
from tqdm import tqdm
for X, y in tqdm(loader, total=len(ds_base)//196):
    pass

 32%|███▏      | 9623/30178 [04:34<28:28, 12.03it/s]

In [22]:
from torch.utils.data import DataLoader, RandomSampler
dataset = PairDataset(root=f"/sietch_colab/kkor/cxt/ts/processed", split="train", mmap=True)
N = len(dataset)
rng = np.random.default_rng(1234)
perm = rng.permutation(N).astype(np.int32)   # ~20 MB for 5M entries
sampler = StaticIndexSampler(perm)


#sampler = RandomSampler(dataset, replacement=False)
loader  = DataLoader(dataset, batch_size=196, sampler=sampler, num_workers=8, pin_memory=True, drop_last=True)

In [23]:
from tqdm import tqdm
for X, y in tqdm(loader):
    pass

  0%|          | 32/28010 [00:29<7:07:42,  1.09it/s]


KeyboardInterrupt: 

In [1]:
import os, numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from cxt.utils import TIMES

def discretize(sequence, population_time):
    idx = np.searchsorted(population_time, sequence, side="right") - 1
    np.clip(idx, 0, len(population_time) - 1, out=idx)
    return idx  # keep as numpy array

class PairDataset(Dataset):
    """
    Each item is ONE pair from ONE TS.
    We prebuild a global index of (X_path, y_path, p_idx) ONCE in file-major order.
    Call shuffle_files(seed) to change the order of FILE BLOCKS without touching disk.
    """
    def __init__(self, root, split="train", mmap=True):
        self.root = root
        self.split = split
        self.mmap = mmap

        self.items = []          # list of (X_path, y_path, p_idx)  [canonical, file-major]
        self._file_spans = []    # list of (start_idx, length) per file, into `items`
        self._file_paths = []    # list of (X_path, y_path) per file  (same order as _file_spans)

        split_dir = os.path.join(root, split)
        # --- build items ONCE in file-major order & record spans ---
        for dirpath, dirnames, filenames in os.walk(split_dir):
            if "X.npy" in filenames and "y.npy" in filenames:
                X_path = os.path.join(dirpath, "X.npy")
                y_path = os.path.join(dirpath, "y.npy")
                # read P cheaply from header once
                Y = np.load(y_path, mmap_mode="r") if mmap else np.load(y_path)
                P = int(Y.shape[0])
                if mmap:
                    del Y
                start = len(self.items)
                for p_idx in range(P):
                    self.items.append((X_path, y_path, p_idx))
                self._file_spans.append((start, P))
                self._file_paths.append((X_path, y_path))

        # mapping state for shuffling FILE order (None means identity)
        self._file_perm = None                  # np.ndarray of file indices
        self._cum_lengths_perm = None           # cumulative lengths in permuted file order

    def __len__(self):
        return len(self.items)

    # ---- cheap file-order shuffle (no disk I/O, no rebuilding items) ----
    def shuffle_files(self, seed=None):
        """
        Shuffle FILE order only. Keeps pairs within each file contiguous.
        O(#files), no np.load calls, no touching `self.items`.
        """
        F = len(self._file_spans)
        if F <= 1:
            self._file_perm = None
            self._cum_lengths_perm = None
            return

        if seed is None:
            perm = np.random.permutation(F)
        else:
            perm = np.random.default_rng(int(seed)).permutation(F)

        # store the permutation and its cumulative lengths
        lengths = np.fromiter((L for (_, L) in self._file_spans), dtype=np.int64, count=F)
        lengths_perm = lengths[perm]
        cum = np.cumsum(lengths_perm, dtype=np.int64)

        self._file_perm = perm.astype(np.int32, copy=False)   # compact
        self._cum_lengths_perm = cum

    # optional: revert to canonical order
    def clear_shuffle(self):
        self._file_perm = None
        self._cum_lengths_perm = None

    # optional epoch API
    def set_epoch(self, epoch: int):
        self.shuffle_files(seed=epoch)

    def _map_logical_to_physical(self, i: int) -> int:
        """
        Map logical index i (0..N-1) in the current FILE order
        to the physical index j into self.items (canonical file-major build).
        """
        if self._file_perm is None:
            return i  # identity (canonical order)

        # find which permuted file block contains i
        k = int(np.searchsorted(self._cum_lengths_perm, i, side="right"))
        prev_cum = 0 if k == 0 else int(self._cum_lengths_perm[k-1])
        offset_in_file = i - prev_cum

        file_idx = int(self._file_perm[k])     # original file index
        start, L = self._file_spans[file_idx]
        # safety clip (should already hold)
        if offset_in_file >= L:
            offset_in_file = L - 1
        return start + offset_in_file

    def __getitem__(self, i):
        # remap i through current file permutation (cheap)
        j = self._map_logical_to_physical(int(i))
        X_path, y_path, p_idx = self.items[j]

        X = np.load(X_path, mmap_mode="r") if self.mmap else np.load(X_path)
        y = np.load(y_path, mmap_mode="r") if self.mmap else np.load(y_path)

        # features
        Xi = torch.tensor(X[p_idx])            # (2, ...)
        Xi = torch.log1p(Xi)

        # labels
        yi = torch.tensor(y[p_idx])
        yi = torch.tensor(discretize(yi, TIMES)).long() + 2
        yi = torch.cat([torch.tensor([1]), yi])

        return Xi, yi


In [5]:
import numpy as np, torch
from torch.utils.data import IterableDataset, DataLoader

class ShuffleBufferDataset(IterableDataset):
    """
    Wraps an indexable dataset `ds` and yields items in near-random order
    using a small FIFO buffer. Reads from disk remain sequential (fast).
    """
    def __init__(self, ds, buffer_size: int = 8192, seed: int = 1234):
        super().__init__()
        self.ds = ds
        self.buffer_size = int(buffer_size)
        self.seed = int(seed)

    def __len__(self):
        return len(self.ds)

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        N = len(self.ds)
        # iterate **sequentially** for disk locality
        src = iter(range(N))
        # fill buffer
        buf = []
        for _ in range(min(self.buffer_size, N)):
            i = next(src, None)
            if i is None: break
            buf.append(i)
        # stream: sample random from buffer, then refill with next sequential index
        while buf:
            j = int(rng.integers(0, len(buf)))
            idx = buf.pop(j)
            yield self.ds[idx]
            nxt = next(src, None)
            if nxt is not None:
                buf.append(nxt)

In [6]:
ds = PairDataset(root="/sietch_colab/kkor/cxt/ts/processed", split="train", mmap=True)
ds.shuffle_files(seed=1234)  # O(#files), no disk I/O
ds = ShuffleBufferDataset(ds, buffer_size=4096*8, seed=1234)
loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=32, drop_last=True, prefetch_factor=4)

In [ ]:
from tqdm import tqdm
for X, y in tqdm(loader):
    pass

  4%|▍         | 2098/47514 [00:33<11:32, 65.54it/s]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f7f8a39c680>>
Traceback (most recent call last):
  File "/home/kkor/miniconda/envs/cxt/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
  4%|▍         | 2104/47514 [00:34<12:19, 61.41it/s]


KeyboardInterrupt: 

: 

In [5]:
X

tensor([[[[[3.5547, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [3.2949, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [2.6387, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [2.3984, 0.0000, 0.6934,  ..., 0.0000, 0.0000, 0.0000],
           [2.6387, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [1.9463, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

          [[4.2891, 0.0000, 0.6934,  ..., 0.0000, 0.0000, 0.0000],
           [4.1602, 0.0000, 0.6934,  ..., 0.0000, 0.0000, 0.0000],
           [4.0234, 0.0000, 0.6934,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [2.8340, 0.0000, 0.6934,  ..., 0.0000, 0.0000, 0.0000],
           [2.6387, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [1.9463, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

          [[5.6875, 1.3867, 2.0801,  ..., 0.0000, 0.0000, 0.0000],
           [5.6680, 1.3867, 2.0801,  ..., 0.0000, 0.0000, 0.0000],
           [5.6367, 1.3867